In [ ]:
import os
import json
from pathlib import Path
from groq import Groq
from rich import print
from litellm import completion
from dotenv import load_dotenv
from price_agent.data.batch import Batch
from price_agent.data.items import Item

load_dotenv(override=True)

d:\ujjwal\the_capstone_project\.venv\Lib\site-packages\huggingface_hub\constants.py:310: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


True

In [2]:
groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))
MODEL = os.environ.get("GROQ_MODEL")

In [ ]:
LITE_MODE = True

# Resolve project root by walking up until pyproject.toml is found.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / "pyproject.toml").exists()), _cwd)

BATCHES_ROOT = PROJECT_ROOT / "data" / "02-preprocessed" / "batches"
BATCH_DIR = BATCHES_ROOT / ("lite" if LITE_MODE else "full")
BATCH_INPUT_FILE = BATCH_DIR / "0_1000.jsonl"
BATCH_RESULTS_FILE = BATCH_DIR / "batch_results.jsonl"

print(f"Project root: {PROJECT_ROOT}")
print(f"Batch directory: {BATCH_DIR}")

In [4]:
username = "ujjwalsingh108"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")

Loaded 22,000 items

In [5]:
print(items[1])

Item(
    title='ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool 
Pirate Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers',
    category='Toys_and_Games',
    price=15.99,
    full='ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool 
Pirate Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers\nArtCreativity\n[\'READY
FOR BOUNTY: Mini pirate treats that spark major grins! This pack comes with a SET OF 24 miniature pirate treasure 
chests, each measuring 1.5”. Eye-catching and with a little room for mini coins and pirate toys, they’re just what 
you need to spruce up any themed party.\', \' PIRATE : With a bit of creativity, you can turn these mini treasure 
chest toys into a compliment-sparking spectacle. Use them to decorate party tables, fashion them into centerpieces,
or use as pirate cake and cupcake toppers kiddies will relish.\', \' THE : Complement these small pirate toys with 
a serving of imagination and a world of fun is yours for the taking. Kids will love using these chests as 
pretend-play pirate toys to create fun where adventure and action reigns supreme.\', \'COOL PARTY FAVORS: Looking 
for pirate party favors? Pirate party goodie bag stuffers or pinata fillers? You’ve found the coolest! They are 
also great as teacher rewards for boys and girls when filled with treats or fun art supplies for your next project.
For ages 3+\', \'BUY RISK-FREE: We fully stand behind our products with a best satisfaction and 100% money-back 
guarantee. Not satisfied with these mini pirate treasure chests? We’ll send you a replacement or issue a full 
refund. Click ‘Add to Cart’ now to fuel the fun risk-free!\']\n{"Brand": "ArtCreativity", "Color": "Gold", 
"Material": "Plastic", "Recommended Uses For Product": "Decorations", "Product Dimensions": "1.5\\"L x 1.5\\"W x 
1.5\\"H", "Capacity": "1.5 Pounds", "Shape": "Rectangular", "Pattern": "Solid", "Number of Items": "24", "Unit 
Count": "24 Count", "Item Weight": "3.84 ounces", "Manufacturer recommended age": "3 years and up", "Manufacturer":
"ArtCreativity"}',
    weight=0.24,
    summary=None,
    prompt=None,
    id=None,
    parent_asin='B08PDSXKFG'
)

In [6]:
print(items[1].id)

None

In [7]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [8]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [9]:
print(items[1].full)

ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool Pirate 
Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers
ArtCreativity
['READY FOR BOUNTY: Mini pirate treats that spark major grins! This pack comes with a SET OF 24 miniature pirate 
treasure chests, each measuring 1.5”. Eye-catching and with a little room for mini coins and pirate toys, they’re 
just what you need to spruce up any themed party.', ' PIRATE : With a bit of creativity, you can turn these mini 
treasure chest toys into a compliment-sparking spectacle. Use them to decorate party tables, fashion them into 
centerpieces, or use as pirate cake and cupcake toppers kiddies will relish.', ' THE : Complement these small 
pirate toys with a serving of imagination and a world of fun is yours for the taking. Kids will love using these 
chests as pretend-play pirate toys to create fun where adventure and action reigns supreme.', 'COOL PARTY FAVORS: 
Looking for pirate party favors? Pirate party goodie bag stuffers or pinata fillers? You’ve found the coolest! They
are also great as teacher rewards for boys and girls when filled with treats or fun art supplies for your next 
project. For ages 3+', 'BUY RISK-FREE: We fully stand behind our products with a best satisfaction and 100% 
money-back guarantee. Not satisfied with these mini pirate treasure chests? We’ll send you a replacement or issue a
full refund. Click ‘Add to Cart’ now to fuel the fun risk-free!']
{"Brand": "ArtCreativity", "Color": "Gold", "Material": "Plastic", "Recommended Uses For Product": "Decorations", 
"Product Dimensions": "1.5\"L x 1.5\"W x 1.5\"H", "Capacity": "1.5 Pounds", "Shape": "Rectangular", "Pattern": 
"Solid", "Number of Items": "24", "Unit Count": "24 Count", "Item Weight": "3.84 ounces", "Manufacturer recommended
age": "3 years and up", "Manufacturer": "ArtCreativity"}

In [10]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT}, 
    {"role": "user", "content": items[1].full}
]

response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(f"{response.choices[0].message.content}\n")
print(f"Input tokens: {response.usage.prompt_tokens}\n")
print(f"Output tokens: {response.usage.completion_tokens}\n")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

22:55:27 - LiteLLM:INFO: utils.py:3831 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
2026-08-09 22:55:27,942 INFO LiteLLM 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
22:55:28 - LiteLLM:INFO: utils.py:1470 - Wrapper: Completed Call, calling success_handler
2026-08-09 22:55:28,627 INFO LiteLLM Wrapper: Completed Call, calling success_handler


Title: Mini Pirate Treasure Chests – 24 Pack  
Category: Party Supplies  
Brand: ArtCreativity  
Description: Set of 24 gold‑finished 1.5‑inch plastic treasure chests perfect for pirate-themed parties.  
Details: Each compact chest holds small prizes and is durable, making them ideal for goodie bags, table décor, or 
imaginative play.

Input tokens: 585

Output tokens: 93

Cost: 0.007 cents

In [11]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [12]:
print(items[1])

Item(
    title='ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool 
Pirate Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers',
    category='Toys_and_Games',
    price=15.99,
    full='ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool 
Pirate Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers\nArtCreativity\n[\'READY
FOR BOUNTY: Mini pirate treats that spark major grins! This pack comes with a SET OF 24 miniature pirate treasure 
chests, each measuring 1.5”. Eye-catching and with a little room for mini coins and pirate toys, they’re just what 
you need to spruce up any themed party.\', \' PIRATE : With a bit of creativity, you can turn these mini treasure 
chest toys into a compliment-sparking spectacle. Use them to decorate party tables, fashion them into centerpieces,
or use as pirate cake and cupcake toppers kiddies will relish.\', \' THE : Complement these small pirate toys with 
a serving of imagination and a world of fun is yours for the taking. Kids will love using these chests as 
pretend-play pirate toys to create fun where adventure and action reigns supreme.\', \'COOL PARTY FAVORS: Looking 
for pirate party favors? Pirate party goodie bag stuffers or pinata fillers? You’ve found the coolest! They are 
also great as teacher rewards for boys and girls when filled with treats or fun art supplies for your next project.
For ages 3+\', \'BUY RISK-FREE: We fully stand behind our products with a best satisfaction and 100% money-back 
guarantee. Not satisfied with these mini pirate treasure chests? We’ll send you a replacement or issue a full 
refund. Click ‘Add to Cart’ now to fuel the fun risk-free!\']\n{"Brand": "ArtCreativity", "Color": "Gold", 
"Material": "Plastic", "Recommended Uses For Product": "Decorations", "Product Dimensions": "1.5\\"L x 1.5\\"W x 
1.5\\"H", "Capacity": "1.5 Pounds", "Shape": "Rectangular", "Pattern": "Solid", "Number of Items": "24", "Unit 
Count": "24 Count", "Item Weight": "3.84 ounces", "Manufacturer recommended age": "3 years and up", "Manufacturer":
"ArtCreativity"}',
    weight=0.24,
    summary=None,
    prompt=None,
    id=1,
    parent_asin='B08PDSXKFG'
)

In [13]:
print(make_jsonl(items[1]))

{"custom_id": "1", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", 
"messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format.
Do not include part numbers.\nTitle: Rewritten short precise title\nCategory: eg Electronics\nBrand: Brand 
name\nDescription: 1 sentence description\nDetails: 1 sentence on features"}, {"role": "user", "content": 
"ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool Pirate 
Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers\nArtCreativity\n['READY FOR 
BOUNTY: Mini pirate treats that spark major grins! This pack comes with a SET OF 24 miniature pirate treasure 
chests, each measuring 1.5\u201d. Eye-catching and with a little room for mini coins and pirate toys, they\u2019re 
just what you need to spruce up any themed party.', ' PIRATE : With a bit of creativity, you can turn these mini 
treasure chest toys into a compliment-sparking spectacle. Use them to decorate party tables, fashion them into 
centerpieces, or use as pirate cake and cupcake toppers kiddies will relish.', ' THE : Complement these small 
pirate toys with a serving of imagination and a world of fun is yours for the taking. Kids will love using these 
chests as pretend-play pirate toys to create fun where adventure and action reigns supreme.', 'COOL PARTY FAVORS: 
Looking for pirate party favors? Pirate party goodie bag stuffers or pinata fillers? You\u2019ve found the coolest!
They are also great as teacher rewards for boys and girls when filled with treats or fun art supplies for your next
project. For ages 3+', 'BUY RISK-FREE: We fully stand behind our products with a best satisfaction and 100% 
money-back guarantee. Not satisfied with these mini pirate treasure chests? We\u2019ll send you a replacement or 
issue a full refund. Click \u2018Add to Cart\u2019 now to fuel the fun risk-free!']\n{\"Brand\": \"ArtCreativity\",
\"Color\": \"Gold\", \"Material\": \"Plastic\", \"Recommended Uses For Product\": \"Decorations\", \"Product 
Dimensions\": \"1.5\\\"L x 1.5\\\"W x 1.5\\\"H\", \"Capacity\": \"1.5 Pounds\", \"Shape\": \"Rectangular\", 
\"Pattern\": \"Solid\", \"Number of Items\": \"24\", \"Unit Count\": \"24 Count\", \"Item Weight\": \"3.84 
ounces\", \"Manufacturer recommended age\": \"3 years and up\", \"Manufacturer\": \"ArtCreativity\"}"}], 
"reasoning_effort": "low"}}

In [ ]:
def make_file(start, end, filename):
    target = Path(filename)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, BATCH_INPUT_FILE)

In [ ]:
with BATCH_INPUT_FILE.open("rb") as f:
    response = groq.files.create(file=f, purpose="batch")
print(response)

FileCreateResponse(
    id='file_01kzkrxx5cea8t9bvs3yzcyw0n',
    bytes=2245925,
    created_at=1786296333,
    filename='0_1000.jsonl',
    object='file',
    purpose='batch',
    size=0,
    md5='ztJlp+YGDPy+OJPFJUgeWw==',
    content_type='application/jsonl'
)

In [17]:
file_id = response.id
print(file_id)

file_01kzkrxx5cea8t9bvs3yzcyw0n

In [18]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
print(response)

BatchCreateResponse(
    id='batch_01kzkrxxnmehbsptrj071h8adv',
    completion_window='24h',
    created_at=1786296334,
    endpoint='/v1/chat/completions',
    input_file_id='file_01kzkrxx5cea8t9bvs3yzcyw0n',
    object='batch',
    status='validating',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=None,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1786382734,
    failed_at=None,
    finalizing_at=None,
    in_progress_at=None,
    metadata=None,
    output_file_id=None,
    request_counts=RequestCounts(completed=0, failed=0, total=0),
    project_id='project_01kag1p6fqfksrpmhwrz3k49tz'
)

In [20]:
result = groq.batches.retrieve(response.id)
print(result)

BatchRetrieveResponse(
    id='batch_01kzkrxxnmehbsptrj071h8adv',
    completion_window='24h',
    created_at=1786296334,
    endpoint='/v1/chat/completions',
    input_file_id='file_01kzkrxx5cea8t9bvs3yzcyw0n',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1786296416,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1786382734,
    failed_at=None,
    finalizing_at=1786296415,
    in_progress_at=1786296338,
    metadata=None,
    output_file_id='file_01kzks0cyvef7b8k0fr4c3zjh4',
    request_counts=RequestCounts(completed=1000, failed=0, total=1000),
    project_id='project_01kag1p6fqfksrpmhwrz3k49tz'
)

In [ ]:
BATCH_DIR.mkdir(parents=True, exist_ok=True)

response = groq.files.content(result.output_file_id)
response.write_to_file(str(BATCH_RESULTS_FILE))
print(f"Saved batch results to {BATCH_RESULTS_FILE}")

Saved batch results to data/02-preprocessed/batches/lite/batch_results.jsonl

In [ ]:
with BATCH_RESULTS_FILE.open("r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary

In [23]:
print(items[1].full)

ArtCreativity Mini Pirate Treasure Chests, Set of 24, 1.5 Inch Plastic Chests with a Gold Finish, Cool Pirate 
Birthday Party Favors Supplies for Kids, Unique Decorations and Goodie Bag Stuffers
ArtCreativity
['READY FOR BOUNTY: Mini pirate treats that spark major grins! This pack comes with a SET OF 24 miniature pirate 
treasure chests, each measuring 1.5”. Eye-catching and with a little room for mini coins and pirate toys, they’re 
just what you need to spruce up any themed party.', ' PIRATE : With a bit of creativity, you can turn these mini 
treasure chest toys into a compliment-sparking spectacle. Use them to decorate party tables, fashion them into 
centerpieces, or use as pirate cake and cupcake toppers kiddies will relish.', ' THE : Complement these small 
pirate toys with a serving of imagination and a world of fun is yours for the taking. Kids will love using these 
chests as pretend-play pirate toys to create fun where adventure and action reigns supreme.', 'COOL PARTY FAVORS: 
Looking for pirate party favors? Pirate party goodie bag stuffers or pinata fillers? You’ve found the coolest! They
are also great as teacher rewards for boys and girls when filled with treats or fun art supplies for your next 
project. For ages 3+', 'BUY RISK-FREE: We fully stand behind our products with a best satisfaction and 100% 
money-back guarantee. Not satisfied with these mini pirate treasure chests? We’ll send you a replacement or issue a
full refund. Click ‘Add to Cart’ now to fuel the fun risk-free!']
{"Brand": "ArtCreativity", "Color": "Gold", "Material": "Plastic", "Recommended Uses For Product": "Decorations", 
"Product Dimensions": "1.5\"L x 1.5\"W x 1.5\"H", "Capacity": "1.5 Pounds", "Shape": "Rectangular", "Pattern": 
"Solid", "Number of Items": "24", "Unit Count": "24 Count", "Item Weight": "3.84 ounces", "Manufacturer recommended
age": "3 years and up", "Manufacturer": "ArtCreativity"}

In [25]:
print(items[1].summary)

Title: Mini Pirate Treasure Chests  
Category: Party Supplies  
Brand: ArtCreativity  
Description: A set of 24 gold‑finish plastic chests, each 1.5 inches, perfect for themed party favors or 
decorations.  
Details: Lightweight, durable plastic with a bright gold coating, ideal for filling with treats or small toy 
treasures.